# Dataset and Benchmark

**Sources used (Hugging Face Hub):**

- Training data: `zwhe99/commonsense_170k` (a 20K slice).
- Benchmarks: `google/boolq`, `Rowan/hellaswag`, `winogrande` (`winogrande_xl`), `allenai/ai2_arc` (`ARC-Easy`, `ARC-Challenge`), `allenai/openbookqa` (`main`), `ybisk/piqa`, `allenai/social_i_qa`.

**Authentication.** Some of the benchmark datasets (and the LLaMA-3.2-1B checkpoint used elsewhere in the project) require a Hugging Face access token. Run `huggingface-cli login` (or `notebook_login()`) before executing the cells below.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes torchao==0.16.0 trl datasets==2.21.0

In [ ]:
from pathlib import Path
import os
from dataclasses import dataclass
from typing import Tuple, Dict, Optional

from datasets import load_dataset

In [ ]:
# Claude Helped making config look nice
@dataclass
class Config:
    # ── Training data ──────────────────────────────────────────────────────────────────
    dataset_name: str   = "zwhe99/commonsense_170k"
    n_train:      int   = 20_000
    n_val:        int   = 500

    warmup_ratio: float = 0.1

    # Two seeds for robustness: LoRA & manual_DoRA are run at both.
    seed:         int   = 67
    alt_seed:     int   = 13

    # ── Evaluation ───────────────────────────────────────────────────────────────────────
    n_bench_samples: int = 1000
    eval_batch_size: int = 8
    eval_steps:      int = 100   # was 250 → finer-grained eval-loss curve
    logging_steps:   int = 5     # was 10  → smoother training-loss curve

    # ── I/O ────────────────────────────────────────────────────────────────────────────────────
    output_dir: str = "/content/gdrive/MyDrive/DoRA_Final_Project/Results"


## Training data: Commonsense170K (20K slice)

In [ ]:
def prepare_datasets(config):
    ds = load_dataset(config.dataset_name, split="train")
    # filter
    def is_valid(ex):
        output = ex.get("output", "")
        return isinstance(output, str) and len(output.strip()) > 0

    ds = ds.filter(is_valid)
    # format
    def format_prompt(ex):
        instruction = ex.get("instruction", "")
        inp = ex.get("input", "")
        output = ex.get("output", "")

        if inp:
            prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
        else:
            prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

        full_text = prompt + output

        return {
            "text": full_text,
            "prompt": prompt,
            "response": output,
        }

    ds = ds.map(format_prompt, remove_columns=ds.column_names)
    # shuffle
    ds = ds.shuffle(seed=config.seed)
    # Split
    split = ds.train_test_split(
        test_size=config.n_val,
        seed=config.seed
    )
    train_data = split["train"].select(range(config.n_train))
    val_data = split["test"]

    print(f"Train: {len(train_data)} | Val: {len(val_data)}")
    return train_data, val_data

In [ ]:
# Run the loader (downloads + caches Commonsense170K to ~/.cache/huggingface).
train_data, val_data = prepare_datasets(config)
print(train_data[0])

## Benchmarks: 8 commonsense-reasoning datasets

In [ ]:
BENCH_DATASETS: Dict[str, Tuple[str, str, Optional[str]]] = {
    "boolq":         ("google/boolq",                   "validation", None),
    "hellaswag":     ("Rowan/hellaswag",                "validation", None),
    "winogrande":    ("winogrande",                     "validation", "winogrande_xl"),
    "arc_easy":      ("allenai/ai2_arc",                "validation", "ARC-Easy"),
    "arc_challenge": ("allenai/ai2_arc",                "validation", "ARC-Challenge"),
    "openbookqa":    ("allenai/openbookqa",             "validation", "main"),
    "piqa":          ("ybisk/piqa",                     "validation", None),
    "siqa":          ("allenai/social_i_qa",            "validation", None),
}

def _load_bench_dataset(repo_id: str, split: str = "validation", config_name: Optional[str] = None):
    if config_name is None:
        return load_dataset(repo_id, split=split, trust_remote_code=True)
    return load_dataset(repo_id, config_name, split=split, trust_remote_code=True)

In [ ]:
# Touch every benchmark so the cache is fully warmed and we can confirm sizes / column names.
for bench_name, (repo_id, split, config_name) in BENCH_DATASETS.items():
    ds = _load_bench_dataset(repo_id, split=split, config_name=config_name)
    print(f"{bench_name:<14} {repo_id:<28} split={split:<12} n={len(ds):<6} columns={ds.column_names}")